# **Telecom RAG Pipeline**


In [1]:
!pip install -q vllm sentence-transformers chromadb pyngrok rank_bm25 huggingface_hub

import os
import uuid
import shutil
import sqlite3
import json
import string
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer
import chromadb
import gc
from tqdm.notebook import tqdm
from rank_bm25 import BM25Okapi


import vllm
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from vllm.sampling_params import SamplingParams
from transformers import AutoTokenizer

from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        print("Authenticating Hugging Face:")
        login(hf_token)
except Exception as e:
    print("HF_TOKEN not found in secrets!")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Setup complete. Using device: {device.upper()}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/6

In [2]:
!pip install pyngrok
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list
!sudo apt-get update -y
!sudo apt-get install ngrok -y


deb https://ngrok-agent.s3.amazonaws.com buster main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packag

In [3]:
# DB_PATH = "./telecom_vector_db"
# SQLITE_PATH = "telecom_sops.db"

import uuid

DB_PATH = f"/tmp/telecom_vector_db_{uuid.uuid4().hex}"
SQLITE_PATH = f"/tmp/telecom_sops_{uuid.uuid4().hex}.db"

# 1. Clean up old runs
if os.path.exists(DB_PATH): shutil.rmtree(DB_PATH)
if os.path.exists(SQLITE_PATH): os.remove(SQLITE_PATH)

# 2. Initialize SQLite (for Document Store)
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sops (
        sop_id TEXT PRIMARY KEY,
        vendor TEXT,
        severity TEXT,
        full_json TEXT
    )
''')
conn.commit()
print("SQLite initialized.")

# 3. Initialize ChromaDB (for Vector Store)
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection(
    name="telecom_sops",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB initialized.")

# 4. Load Embedding Model
print("Loading SOTA Embedding Model: BAAI/bge-base-en-v1.5")
embedding_func = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)


SQLite initialized.
ChromaDB initialized.
Loading SOTA Embedding Model: BAAI/bge-base-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
print("Loading structured SOPs and rewriting for dense embeddings:")

with open("telecom_sops.json", "r") as f:
    dataset = json.load(f)

bm25_tokenized_corpus = []
batch_docs = []
batch_metadatas = []
batch_ids = []

seen_ids = set()

def simple_tokenize(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation)).split()

for sop in tqdm(dataset, desc="Indexing"):
    sop_id = sop["sop_id"]

    if sop_id in seen_ids:
        continue
    seen_ids.add(sop_id)

    vendor = sop.get("vendor", "Unknown")
    severity = sop.get("severity", "UNKNOWN")

    # Conversational Data Representation
    term = sop.get('title', '').replace('SOP for ', '').replace(' Disruption', '')
    prose_content = f"This is a {severity} severity Standard Operating Procedure (SOP) for {vendor} equipment. "
    prose_content += f"It resolves disruptions caused by {term}. The procedure involves the following steps: "

    steps_text = " ".join([f"Step {s.get('step_number')}: {s.get('action')}. Execute command: `{s.get('command')}`." for s in sop.get('steps', [])])
    prose_content += steps_text

    # Override the original rigid search_content
    sop["search_content"] = prose_content

    # 1. Insert into SQLite
    cursor.execute(
        "INSERT OR REPLACE INTO sops (sop_id, vendor, severity, full_json) VALUES (?, ?, ?, ?)",
        (sop_id, vendor, severity, json.dumps(sop))
    )

    # 2. Prepare for Vector DB & BM25
    bm25_tokenized_corpus.append(simple_tokenize(prose_content))
    batch_docs.append(prose_content)
    batch_ids.append(sop_id)
    batch_metadatas.append({"vendor": vendor, "severity": severity})

conn.commit()

# 3. Insert into ChromaDB
embeddings = embedding_func.encode(batch_docs, normalize_embeddings=True).tolist()
collection.add(
    documents=batch_docs,
    embeddings=embeddings,
    metadatas=batch_metadatas,
    ids=batch_ids
)

print(f"Indexed {collection.count()} SOPs with optimized prose embeddings.")

print("Building BM25 Index:")
bm25 = BM25Okapi(bm25_tokenized_corpus)
del bm25_tokenized_corpus
gc.collect()


Loading structured SOPs and rewriting for dense embeddings:


Indexing:   0%|          | 0/1192 [00:00<?, ?it/s]

Indexed 1192 SOPs with optimized prose embeddings.
Building BM25 Index:


88

In [5]:
import torch

print("\nLoading vLLM Model: hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4:")
import vllm
import os
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        print("Authenticating Hugging Face:")
        login(hf_token)
except Exception as e:
    print("HF_TOKEN not found in secrets!")

from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from transformers import AutoTokenizer

engine_args = AsyncEngineArgs(
    model="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    quantization="awq_marlin",
    tensor_parallel_size=1,
    max_model_len=8192,
    gpu_memory_utilization=0.7,
    enable_prefix_caching=True,
    enforce_eager=False
)
engine = AsyncLLMEngine.from_engine_args(engine_args)

model_id = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("vLLM Async Engine Loaded Successfully.")

print("\nLoading SOTA Cross-Encoder: BAAI/bge-reranker-base")
from sentence_transformers import CrossEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker = CrossEncoder('BAAI/bge-reranker-base', device=device)


Loading vLLM Model: hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4:
Authenticating Hugging Face:


config.json: 0.00B [00:00, ?B/s]

INFO 04-22 10:56:30 [model.py:549] Resolved architecture: LlamaForCausalLM
INFO 04-22 10:56:30 [model.py:1678] Using max model len 8192
INFO 04-22 10:56:31 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 04-22 10:56:31 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=2048.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [6]:
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 59.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.4
    Uninstalling transformers-5.5.4:
      Successfully uninstalled transformers-5.5.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.19.1 requires transformers!=5.0.*,!=5.1.*,!=5.2.*,!=5.3.*,!=5.4.*,!=5.5.0,>=4.56.0, but you have transformers 5.1.0 which is incompatible.


In [7]:
from gliner import GLiNER
import torch

print("Loading GLiNER for localized, zero-shot metadata extraction")

# explicitly bind GLiNER to the CPU to save VRAM for vLLM
device_gliner = 'cpu'

# Load the model weights and move to CPU
ner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1").to(device_gliner)

def check_heuristic_negation(query, entity_start):
    negation_tokens = ["ignore", "skip", "not", "exclude", "except"]
    preceding_words = query[:entity_start].lower().strip().split()
    last_words = preceding_words[-4:]
    return any(tok in last_words for tok in negation_tokens)

def extract_metadata_gliner(query):
    """Extracts vendor and severity dynamically without an external API."""
    labels = ["telecom vendor", "severity level"]

    # Predict entities directly from the raw string
    entities = ner_model.predict_entities(query, labels)

    # Use a dictionary to guarantee the casing perfectly matches the generated JSON metadata
    VENDOR_MAP = {
        "cisco": "Cisco",
        "juniper": "Juniper",
        "nokia": "Nokia",
        "ericsson": "Ericsson",
        "huawei": "Huawei",
        "zte": "ZTE",
        "mavenir": "Mavenir",
        "samsung networks": "Samsung Networks",
        "samsung": "Samsung Networks",
        "ciena": "Ciena",
        "palo alto": "Palo Alto",
        "fortinet": "Fortinet",
        "check point": "Check Point",
        "f5": "F5"
    }

    filters = {}
    for ent in entities:
        text_val = ent["text"].lower()
        label = ent["label"]
        start_idx = ent["start"]

        # Clean and normalize the extracted text to match ChromaDB metadata
        if label == "telecom vendor":
            for key, exact_vendor in VENDOR_MAP.items():
                if key in text_val:
                    if check_heuristic_negation(query, start_idx):
                        filters["vendor"] = {"$ne": exact_vendor}
                    else:
                        filters["vendor"] = exact_vendor
                    break
        elif label == "severity level":
            for s in ["critical", "major", "minor", "warning"]:
                if s in text_val:
                    filters["severity"] = s.upper()
                    break

    if not filters:
        return None
    elif len(filters) == 1:
        return filters
    else:
        return {"$and": [{k: v} for k, v in filters.items()]}


[ERROR] `cache_position` is part of T5Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/t5/modeling_t5.py.
[ERROR] `cache_position` is part of T5ForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/t5/modeling_t5.py.
Loading GLiNER for localized, zero-shot metadata extraction


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [8]:
import numpy as np
import json
import uuid
from vllm.sampling_params import SamplingParams

def assemble_prompt(sys_prompt, history, context_string, query):
    prompt = f"<|start_header_id|>system<|end_header_id|>\n{sys_prompt}<|eot_id|>\n"
    for turn in history:
        query_text = turn.get('query', '')
        resp_text = turn.get('response', '')
        prompt += f"<|start_header_id|>user<|end_header_id|>\n{query_text}<|eot_id|>\n"
        if resp_text:
            prompt += f"<|start_header_id|>assistant<|end_header_id|>\n{resp_text}<|eot_id|>\n"
    prompt += f"<|start_header_id|>user<|end_header_id|>\nContext SOPs:\n{context_string}\n\nUser Issue: {query}<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n"
    return prompt


async def process_query(query, history=None):
    if history is None: history = []

    # Phase 3: Extract Metadata Filters using Local GLiNER
    clean_query = query
    chroma_filters = extract_metadata_gliner(clean_query)
    print(f"  [Diagnostics] Router applied filters: {chroma_filters}")

    # Dense Retrieval
    query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {clean_query}"], normalize_embeddings=True).tolist()

    dense_res = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        where=chroma_filters
    )

    if not dense_res['ids'][0]:
        return "No matching SOPs found for this specific hardware/fault.", []

    dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}

    # Sparse Retrieval (BM25)
    tokenized_query = simple_tokenize(clean_query)
    sparse_scores = bm25.get_scores(tokenized_query)
    top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

    # Phase 1: Tuned Reciprocal Rank Fusion (RRF) for dense datasets
    fusion_scores = {}
    k = 20 # Aggressively favors the top hits to reduce noise before the Cross-Encoder

    for rank, (doc_id, _) in enumerate(dense_hits.items()):
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    for rank, idx in enumerate(top_sparse_indices):
        doc_id = batch_ids[idx]
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:10]
    top_ids = [doc_id for doc_id, _ in sorted_candidates]

    # Fetch FULL JSON from decoupled SQLite storage
    placeholders = ','.join(['?'] * len(top_ids))
    cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
    rows = cursor.fetchall()

    id_to_json = {row[0]: json.loads(row[1]) for row in rows}
    candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]

    # Reranking via Cross-Encoder
    pairs = [[query, doc["search_content"]] for doc in candidate_docs]
    scores = reranker.predict(pairs)
    ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)

    # Phase 1: Expanded Context Window (Taking Top 7 instead of Top 3)
    final_sops = [doc for doc, score in ranked_final][:5]

    # Format Context for Operational Safety
    context_blocks = []
    for sop in final_sops:
        block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
        block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
        for step in sop.get('steps', []):
            block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
        context_blocks.append(block)

    context_string = "\n\n".join(context_blocks)

    # Phase 1: Strict XML Chain-of-Thought Prompting
    # Phase 1: Robust Chain-of-Thought Prompting for 4-bit Models
    sys_prompt = (
        "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
        "You must follow these instructions exactly:\n"
        "1. Identify the SINGLE most relevant SOP for the user's issue.\n"
        "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
        "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
        "4. In your final answer, do NOT generate artificial warnings unless the chosen SOP explicitly states them.\n"
        "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command (e.g., `clear ip bgp *` [SOP-123])."
    )

    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Context SOPs:\n{context_string}\n\nUser Issue: {query}"},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Asynchronous vLLM Generation
    sampling_params = SamplingParams(temperature=0.0, max_tokens=2048)
    req_id = uuid.uuid4().hex

    generated_text = ""
    async for output in engine.generate(prompt, sampling_params, req_id):
        generated_text = output.outputs[0].text

    # Fail proof extraction: Split the string and take everything after the marker
    if "FINAL_ANSWER:" in generated_text:
        answer = generated_text.split("FINAL_ANSWER:")[-1].strip()
    else:
        # Emergency fallback if the model completely ignores formatting
        answer = generated_text.strip()

    return answer, final_sops




In [9]:
async def process_query_stream(query, history=None):
    if history is None: history = []
    try:
        try:
            method = wandb.config.get("negation_method", "heuristic")
        except Exception:
            method = "heuristic"
        clean_query = query
        if method == "llm":
            clean_query = await llm_query_rewrite_for_negation(query)

        chroma_filters = extract_metadata_gliner(clean_query)
        query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {clean_query}"], normalize_embeddings=True).tolist()

        dense_res = collection.query(query_embeddings=query_emb, n_results=10, where=chroma_filters)
        if not dense_res['ids'][0]:
            yield json.dumps({"type": "content", "content": "No matching SOPs found."})
            yield json.dumps({"type": "done"})
            return

        dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}
        tokenized_query = simple_tokenize(clean_query)
        sparse_scores = bm25.get_scores(tokenized_query)
        top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

        fusion_scores = {}
        k = 20
        for rank, (doc_id, _) in enumerate(dense_hits.items()): fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)
        for rank, idx in enumerate(top_sparse_indices):
            doc_id = batch_ids[idx]
            fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

        sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        top_ids = [doc_id for doc_id, _ in sorted_candidates]

        placeholders = ','.join(['?'] * len(top_ids))
        cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
        rows = cursor.fetchall()
        id_to_json = {row[0]: json.loads(row[1]) for row in rows}
        candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]

        pairs = [[query, doc["search_content"]] for doc in candidate_docs]
        scores = reranker.predict(pairs)
        ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)
        final_sops = [doc for doc, score in ranked_final][:7]

        sources_data = []
        for sop in final_sops:
            sources_data.append({
                "sop_id": sop['sop_id'],
                "vendor": sop.get('vendor', 'Unknown'),
                "severity": sop.get('severity', 'UNKNOWN'),
                "title": sop.get('title', 'Untitled')
            })
        yield json.dumps({"type": "sources", "sources": sources_data})

        context_blocks = []
        for sop in final_sops:
            block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
            block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
            for step in sop.get('steps', []):
                block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
            context_blocks.append(block)
        context_string = "\n\n".join(context_blocks)

        sys_prompt = (
            "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
            "You must follow these instructions exactly:\n"
            "1. Identify SINGLE most relevant SOP for user's issue.\n"
            "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
            "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
            "4. In your final answer, do NOT generate artificial warnings unless chosen SOP explicitly states them.\n"
            "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command."
        )

        prompt = assemble_prompt(sys_prompt, history, context_string, query)

        sampling_params = SamplingParams(temperature=0.0, max_tokens=2048)
        req_id = uuid.uuid4().hex

        last_yielded_len = 0
        async for output in engine.generate(prompt, sampling_params, req_id):
            text = output.outputs[0].text
            new_text = text[last_yielded_len:]
            if new_text:
                yield json.dumps({"type": "content", "content": new_text})
                last_yielded_len = len(text)

        yield json.dumps({"type": "done"})

    except Exception as e:
        yield json.dumps({"type": "error", "error": str(e)})




In [10]:
!pip install -qU langchain-google-genai
!pip install ragas
!pip install wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully

In [11]:
import os
import pandas as pd
from tqdm.notebook import tqdm
from google.colab import userdata

# 1. Langchain Google GenAI Integrations
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))
config = {
    'embedding_model': 'nomic-embed-text-v1.5',
    'retriever_top_k': 5,
    'rrf_k': 20,
    'llm_temperature': 0.0,
    'negation_method': 'heuristic'
}
wandb.init(project='netrestore-rag-eval', config=config)

# 2. Ragas v0.4+ Imports
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

test_set = [
  {
    "question": "We are seeing massive 5G UE registration failures across the network. Subscribers can't even attach to the network. What function should I investigate?",
    "ground_truth": "Severity: CRITICAL. You must investigate the AMF (Access and Mobility Function) for 5G UE registration failures. STRICT WARNING: Do NOT confuse this with the SMF (Session Management Function). The SMF handles data sessions, but the core issue for initial attach and registration failures lies with the AMF."
  },
  {
    "question": "I need to run a live packet capture debug on the enterprise firewall to see why traffic is dropping. What command should I use? I need to make absolutely sure I don't drop the security policy.",
    "ground_truth": "Severity: CRITICAL. To safely debug firewall traffic, you must use a command like `fw control zdebug`. STRICT WARNING: Do absolutely NOT use `fw unloadlocal`. That command will instantly unload the firewall policy, leaving the device wide open to attack or completely isolating it."
  },
  {
    "question": "Mobile sessions are successfully establishing in the core, so signaling looks completely fine, but absolutely zero user data is actually flowing. Which protocol is failing?",
    "ground_truth": "Severity: CRITICAL. If signaling is active but user data is dropped, you must investigate GTP-U (GTP User Plane). STRICT WARNING: Do NOT confuse this with GTP-C (GTP Control Plane). Troubleshooting GTP-C will not resolve this, as the control plane sessions are already successfully established."
  },
  {
    "question": "Inter-cell handovers are failing between adjacent Nokia base stations, causing dropped calls when users drive, though the towers still have connectivity to the core. Which interface needs to be troubleshot?",
    "ground_truth": "Severity: CRITICAL. To resolve inter-cell handover failures between adjacent base stations, you must troubleshoot the X2 Interface. STRICT WARNING: Do NOT confuse this with the S1 Interface. Bouncing the S1 Interface will cause a total loss of core connectivity for all subscribers on those base stations."
  },
  {
    "question": "Our enterprise edge firewalls are experiencing a Split-Brain scenario where both devices assume the Active role. What is the impact of this?",
    "ground_truth": "Severity: CRITICAL. A Split-Brain HA scenario causes massive IP conflicts and ARP poisoning across the network. STRICT WARNING: Do NOT confuse this with an HA Sync Failure. A sync failure means configurations don't match, but a Split-Brain means both firewalls are actively fighting to route the same traffic."
  },
  {
    "question": "We have a critical fiber link failure on the Ciena transport ring. The optics team thinks it's a wavelength mismatch. Should we check the DWDM or CWDM configuration to avoid taking down the whole link?",
    "ground_truth": "Severity: CRITICAL. You must investigate the DWDM configuration for optical wavelength mismatches causing link failures. STRICT WARNING: Do NOT confuse this with CWDM, which pertains to reduced capacity and different spectrum spacing, but not this specific major optical failure."
  },
  {
    "question": "I need to drop all active data-plane IPsec VPN tunnels momentarily on the Check Point gateway. Should I use 'clear crypto ipsec sa' or 'clear crypto isakmp'?",
    "ground_truth": "Severity: MAJOR. To drop all active data-plane IPsec VPN tunnels momentarily, you must target `clear crypto ipsec sa`. STRICT WARNING: Do NOT confuse this with `clear crypto isakmp`. Running the wrong command will impact Phase 1 tunnels differently."
  },
  {
    "question": "We have major subscriber authentication failures in our LTE/5G interworking setup on the Samsung core. Is this an issue with the HSS or the UDM?",
    "ground_truth": "Severity: CRITICAL. For subscriber authentication failures in LTE/5G interworking, you must investigate the HSS. STRICT WARNING: Do NOT confuse this with the UDM. While the UDM handles 5G profiles, confusing the two will lead to subscriber profile inconsistency."
  },
  {
    "question": "There is a severe routing domain partition happening in the backbone on our ZTE routers. Do we need to look at OSPF Area 0 or the stub areas?",
    "ground_truth": "Severity: CRITICAL. To resolve a routing domain partition in the backbone, you must investigate OSPF Area 0. STRICT WARNING: Do NOT confuse this with a Non-Backbone Area or Stub Area. Misconfiguring this will further tear down backbone routing."
  },
  {
    "question": "Customers are complaining that their legitimate e-commerce traffic is being instantly dropped due to false positives on the F5 load balancer. Did someone put the WAF in Block Mode or Transparent Mode?",
    "ground_truth": "Severity: CRITICAL. If legitimate customer e-commerce traffic is being instantly dropped due to false positives, the WAF is likely in WAF Block Mode. STRICT WARNING: Do NOT confuse this with WAF Transparent Mode, which logs traffic without actively blocking it."
  }
]

# 3. SETUP MODELS & API via Langchain
api_key = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = api_key

print("Initializing Gemma 3 27B and Embeddings:")

# Using Langchain to handle the proper .invoke() / .chat() translations for Ragas
evaluator_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemma-3-27b-it",
        temperature=0.0,
        max_retries=5
    )
)

# Text-embedding-004 for metric evaluations
evaluator_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
)

# 4. BUILD RAGAS DATASET
print("Running inferences and building the EvaluationDataset:")
samples = []

for item in tqdm(test_set):
    query = item["question"]
    gt = item["ground_truth"]

    # Execute local async model generation pipeline
    ans, srcs = await process_query(query)

    # Extract strings from the ChromaDB retrieved chunks
    retrieved_ctx = [doc["search_content"] for doc in srcs]



    # Creating the strict v0.4 SingleTurnSample object
    sample = SingleTurnSample(
        user_input=query,
        response=ans,
        retrieved_contexts=retrieved_ctx,
        reference=gt
    )
    samples.append(sample)

eval_dataset = EvaluationDataset(samples=samples)

# 5. CONFIGURE & RUN EVALUATION
# Rate limiting configuration to protect against API throttling
rate_limit_config = RunConfig(
    max_workers=2,
    max_retries=10,
    max_wait=30
)

print("\nEvaluating pipeline using Gemma 3 27B:")
results = evaluate(
    dataset=eval_dataset,
    metrics=[
        ContextPrecision(),
        ContextRecall(),
        Faithfulness(),
        # AnswerRelevancy(),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=rate_limit_config
)

df_results = results.to_pandas()
display(df_results)

wandb.log({
    'faithfulness': df_results['faithfulness'].mean(),
    # 'answer_relevancy': df_results['answer_relevancy'].mean(),
    'context_precision': df_results['context_precision'].mean(),
    'context_recall': df_results['context_recall'].mean()
})



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sameerthakur0099 (sameerthakur0099-international-institute-of-information-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [google.genai, mcp, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_3816/2726409485.py:22: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipyker

Initializing Gemma 3 27B and Embeddings:


/tmp/ipykernel_3816/2726409485.py:82: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


Running inferences and building the EvaluationDataset:


/tmp/ipykernel_3816/2726409485.py:91: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


  0%|          | 0/10 [00:00<?, ?it/s]

  [Diagnostics] Router applied filters: None
WARNING 04-22 11:04:17 [input_processor.py:235] Passing raw prompts to InputProcessor is deprecated and will be removed in v0.18. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().
  [Diagnostics] Router applied filters: None
  [Diagnostics] Router applied filters: None
  [Diagnostics] Router applied filters: {'vendor': 'Nokia'}
  [Diagnostics] Router applied filters: None
  [Diagnostics] Router applied filters: None
  [Diagnostics] Router applied filters: None
  [Diagnostics] Router applied filters: {'vendor': 'Samsung Networks'}
  [Diagnostics] Router applied filters: {'vendor': 'ZTE'}
  [Diagnostics] Router applied filters: None

Evaluating pipeline using Gemma 3 27B:


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness
0,We are seeing massive 5G UE registration failu...,[This is a CRITICAL severity Standard Operatin...,Verify AMF process CPU utilization and identif...,Severity: CRITICAL. You must investigate the A...,1.000000,1.00,1.000000
1,I need to run a live packet capture debug on t...,[This is a CRITICAL severity Standard Operatin...,To run a live packet capture on the enterprise...,Severity: CRITICAL. To safely debug firewall t...,0.477778,1.00,0.500000
2,Mobile sessions are successfully establishing ...,[This is a CRITICAL severity Standard Operatin...,"To troubleshoot the issue, follow the steps ou...",Severity: CRITICAL. If signaling is active but...,0.950000,1.00,0.950000
3,Inter-cell handovers are failing between adjac...,[This is a MAJOR severity Standard Operating P...,Verify X2 interface status and identify affect...,Severity: CRITICAL. To resolve inter-cell hand...,0.250000,1.00,1.000000
4,Our enterprise edge firewalls are experiencing...,[This is a CRITICAL severity Standard Operatin...,To resolve the Split-Brain scenario where both...,Severity: CRITICAL. A Split-Brain HA scenario ...,1.000000,1.00,1.000000
5,We have a critical fiber link failure on the C...,[This is a CRITICAL severity Standard Operatin...,To resolve the critical fiber link failure on ...,Severity: CRITICAL. You must investigate the D...,1.000000,1.00,0.888889
6,I need to drop all active data-plane IPsec VPN...,[This is a MAJOR severity Standard Operating P...,`clear crypto ipsec sa` [SOP-AD8F0345],Severity: MAJOR. To drop all active data-plane...,1.000000,0.75,1.000000
7,We have major subscriber authentication failur...,[This is a CRITICAL severity Standard Operatin...,"To troubleshoot the issue, follow SOP-FAF8EEC3...",Severity: CRITICAL. For subscriber authenticat...,1.000000,1.00,0.956522
8,There is a severe routing domain partition hap...,[This is a CRITICAL severity Standard Operatin...,Verify OSPF process status on all core routers...,Severity: CRITICAL. To resolve a routing domai...,1.000000,1.00,1.000000
9,Customers are complaining that their legitimat...,[This is a CRITICAL severity Standard Operatin...,"To resolve the issue, follow the steps outline...",Severity: CRITICAL. If legitimate customer e-c...,1.000000,1.00,0.958333


In [12]:
import time
import torch
import pandas as pd
import numpy as np
import gc
import json
import uuid
from vllm.sampling_params import SamplingParams

async def run_secure_noc_benchmark(test_queries):
    # Initialize CUDA graphs and KV-cache allocations
    await process_query("dummy warm up query for cisco bgp")

    print(f"\nStarting Benchmark on {len(test_queries)} queries...")
    results = []

    # Force garbage collection and reset PyTorch VRAM trackers
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for i, query in enumerate(test_queries):
        print(f"  -> Profiling Query {i+1}/{len(test_queries)}: {query[:50]}...")
        metrics = {"Query": query[:60] + "..."}

        # 1. RETRIEVAL PHASE
        t0 = time.time()

        chroma_filters = extract_metadata_gliner(query)
        query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {query}"], normalize_embeddings=True).tolist()

        dense_res = collection.query(query_embeddings=query_emb, n_results=10, where=chroma_filters)

        tokenized_query = simple_tokenize(query)
        sparse_scores = bm25.get_scores(tokenized_query)
        top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

        # RRF logic
        fusion_scores = {}
        k = 20

        if dense_res['ids'] and dense_res['ids'][0]:
            dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}
            for rank, (doc_id, _) in enumerate(dense_hits.items()):
                fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

        for rank, idx in enumerate(top_sparse_indices):
            doc_id = batch_ids[idx]
            fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

        sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:10]
        top_ids = [doc_id for doc_id, _ in sorted_candidates]

        placeholders = ','.join(['?'] * len(top_ids))
        if top_ids:
            cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
            rows = cursor.fetchall()
            id_to_json = {row[0]: json.loads(row[1]) for row in rows}
            candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]
        else:
            candidate_docs = []

        t1 = time.time()
        metrics["Retrieval_Time_sec"] = round(t1 - t0, 3)

        # 2. CROSS-ENCODER RERANKING PHASE
        t2 = time.time()
        if candidate_docs:
            pairs = [[query, doc["search_content"]] for doc in candidate_docs]
            scores = reranker.predict(pairs)
            ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)
            final_sops = [doc for doc, score in ranked_final][:5]
        else:
            final_sops = []

        t3 = time.time()
        metrics["Rerank_Time_sec"] = round(t3 - t2, 3)
        metrics["Retrieved_SOPs_Count"] = len(final_sops)

        # Tracking SOP lengths
        avg_sop_len = np.mean([len(sop.get("search_content", "")) for sop in final_sops]) if final_sops else 0
        metrics["Avg_SOP_Length_chars"] = int(avg_sop_len)

        # 3. PROMPT CONSTRUCTION
        context_blocks = []
        for sop in final_sops:
            block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
            block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
            for step in sop.get('steps', []):
                block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
            context_blocks.append(block)

        context_string = "\n\n".join(context_blocks)

        sys_prompt = (
            "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
            "You must follow these instructions exactly:\n"
            "1. Identify the SINGLE most relevant SOP for the user's issue.\n"
            "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
            "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
            "4. In your final answer, do NOT generate artificial warnings unless the chosen SOP explicitly states them.\n"
            "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command (e.g., `clear ip bgp *` [SOP-123])."
        )

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Context SOPs:\n{context_string}\n\nUser Issue: {query}"},
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_tokens = len(tokenizer.encode(prompt))
        metrics["Prompt_Tokens"] = prompt_tokens

        # 4. vLLM GENERATION & TTFT PROFILING
        sampling_params = SamplingParams(temperature=0.0, max_tokens=2048)
        req_id = uuid.uuid4().hex

        t4 = time.time()
        generated_text = ""
        first_token_time = None

        # Asynchronous stream consumption to measure TTFT
        async for output in engine.generate(prompt, sampling_params, req_id):
            if first_token_time is None:
                first_token_time = time.time()
            generated_text = output.outputs[0].text

        t5 = time.time()

        ttft = first_token_time - t4 if first_token_time else 0
        gen_time = t5 - first_token_time if first_token_time else 0

        metrics["TTFT_sec"] = round(ttft, 3)
        metrics["Generation_Time_sec"] = round(gen_time, 3)

        # Calculate true End-to-End Latency
        metrics["E2E_Latency_sec"] = round((t1 - t0) + (t3 - t2) + ttft + gen_time, 3)

        gen_tokens = len(tokenizer.encode(generated_text))
        metrics["Generated_Tokens"] = gen_tokens
        metrics["Tokens_Per_Sec"] = round(gen_tokens / gen_time, 2) if gen_time > 0 else 0

        # 5. HARDWARE PROFILING
        peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
        metrics["Peak_VRAM_GB"] = round(peak_vram, 2)

        results.append(metrics)

    df_results = pd.DataFrame(results)

    # Reorder columns for better readability
    cols = ['Query', 'E2E_Latency_sec', 'Tokens_Per_Sec', 'TTFT_sec', 'Retrieval_Time_sec', 'Rerank_Time_sec', 'Retrieved_SOPs_Count', 'Prompt_Tokens', 'Generated_Tokens', 'Peak_VRAM_GB']
    df_results = df_results[cols]

    return df_results

# EXPANDED TEST
expanded_test_queries = [
    # Routing & Core Networking
    "Urgent: I'm seeing stale routes on the US-East Core Cisco. I need to force a route refresh immediately but CANNOT drop active traffic. What do I type?",
    "There is a severe routing domain partition happening in the backbone on our ZTE routers. Do we need to look at OSPF Area 0 or the stub areas?",

    # Security & Firewalls
    "We've got enterprise clients complaining about their Check Point IPsec VPN tunnels flapping. Should I use 'clear crypto ipsec sa' or 'clear crypto isakmp'?",
    "Our Palo Alto edge firewalls are experiencing a Split-Brain scenario where both devices assume the Active role. What is the impact of this?",
    "I need to run a live packet capture debug on the Fortinet firewall to see why traffic is dropping. What command should I use to safely check traffic?",

    # 5G & Mobile Core
    "Active mobile handovers are failing across the EU-Central RAN. Subscribers are dropping calls when driving. What Nokia gateway do I check?",
    "We are seeing massive 5G UE registration failures across the Ericsson network. Subscribers can't even attach. What function should I investigate?",
    "Mobile sessions are successfully establishing in the core, so signaling looks fine, but absolutely zero user data is actually flowing. Which protocol is failing?",
    "We have major subscriber authentication failures in our LTE/5G interworking setup on the Samsung core. Is this an issue with the HSS or the UDM?",

    # Load Balancing & WAF
    "Customers are complaining that their legitimate e-commerce traffic is being instantly dropped due to false positives on the F5 load balancer.",

    # Optical / Transport
    "We have a critical fiber link failure on the Ciena transport ring. The optics team thinks it's a wavelength mismatch. Should we check the DWDM or CWDM config?"
]

# AWAIT the async function call
benchmark_df = await run_secure_noc_benchmark(expanded_test_queries)

display(benchmark_df.style.background_gradient(subset=['E2E_Latency_sec', 'TTFT_sec'], cmap='YlOrRd'))


  [Diagnostics] Router applied filters: {'vendor': 'Cisco'}

Starting Benchmark on 11 queries...
  -> Profiling Query 1/11: Urgent: I'm seeing stale routes on the US-East Cor...
  -> Profiling Query 2/11: There is a severe routing domain partition happeni...
  -> Profiling Query 3/11: We've got enterprise clients complaining about the...
  -> Profiling Query 4/11: Our Palo Alto edge firewalls are experiencing a Sp...
  -> Profiling Query 5/11: I need to run a live packet capture debug on the F...
  -> Profiling Query 6/11: Active mobile handovers are failing across the EU-...
  -> Profiling Query 7/11: We are seeing massive 5G UE registration failures ...
  -> Profiling Query 8/11: Mobile sessions are successfully establishing in t...
  -> Profiling Query 9/11: We have major subscriber authentication failures i...
  -> Profiling Query 10/11: Customers are complaining that their legitimate e-...
  -> Profiling Query 11/11: We have a critical fiber link failure on the Ciena...


,Query,E2E_Latency_sec,Tokens_Per_Sec,TTFT_sec,Retrieval_Time_sec,Rerank_Time_sec,Retrieved_SOPs_Count,Prompt_Tokens,Generated_Tokens,Peak_VRAM_GB
0,Urgent: I'm seeing stale routes on the US-East Core Cisco. I...,55.823000,38.100000,1.302000,0.453000,0.286000,5,2163,2049,1.620000
1,There is a severe routing domain partition happening in the ...,4.808000,38.790000,0.046000,0.259000,0.275000,5,2059,164,1.620000
2,We've got enterprise clients complaining about their Check P...,14.793000,37.920000,1.447000,0.339000,0.295000,5,2263,482,1.620000
3,Our Palo Alto edge firewalls are experiencing a Split-Brain ...,13.994000,37.660000,1.309000,0.307000,0.296000,5,2063,455,1.620000
4,I need to run a live packet capture debug on the Fortinet fi...,9.920000,37.420000,1.285000,0.322000,0.298000,5,1990,300,1.620000
5,Active mobile handovers are failing across the EU-Central RA...,5.397000,37.450000,1.404000,0.236000,0.286000,5,2156,130,1.620000
6,We are seeing massive 5G UE registration failures across the...,18.294000,36.390000,1.555000,0.263000,0.289000,5,2339,589,1.620000
7,"Mobile sessions are successfully establishing in the core, s...",6.254000,36.750000,1.580000,0.245000,0.292000,5,2363,152,1.620000
8,We have major subscriber authentication failures in our LTE/...,15.729000,37.090000,0.219000,1.832000,0.520000,5,2235,488,1.620000
9,Customers are complaining that their legitimate e-commerce t...,13.878000,36.780000,1.673000,0.320000,0.302000,5,2515,426,1.620000


In [13]:
import time

negation_test_set = [
    {"query": "BGP routes are constantly flapping on the core network. Ignore the ZTE documentation.", "expected_vendor": None, "excluded_vendor": "ZTE"},
    {"query": "The Cisco router is operating normally, but the DWDM link is down. Skip the Cisco docs.", "expected_vendor": None, "excluded_vendor": "Cisco"},
    {"query": "How do I troubleshoot OSPF neighbor state stuck in EXSTART? Give me everything except Juniper.", "expected_vendor": None, "excluded_vendor": "Juniper"},
    {"query": "I am not looking for Nokia instructions right now. Give me the Cisco steps for resetting the AMF interface.", "expected_vendor": "Cisco", "excluded_vendor": "Nokia"},
    {"query": "Fix corrupted BGP table on Juniper router.", "expected_vendor": "Juniper", "excluded_vendor": None}
]

async def evaluate_negation_routing():
    print(f"\n--- Running Negation Benchmark (Heuristic Window) ---")

    for item in negation_test_set:
        query = item["query"]
        expected_vendor = item["expected_vendor"]
        excluded_vendor = item["excluded_vendor"]

        start_time = time.perf_counter()

        # Execute extraction logic
        clean_query = query
        chroma_filters = extract_metadata_gliner(clean_query)

        extraction_latency_ms = (time.perf_counter() - start_time) * 1000

        # Evaluate filter correctness
        extracted_vendor_val = None
        if chroma_filters:
            if "vendor" in chroma_filters:
                extracted_vendor_val = chroma_filters["vendor"]
            elif "$and" in chroma_filters:
                for filter_obj in chroma_filters["$and"]:
                    if "vendor" in filter_obj:
                        extracted_vendor_val = filter_obj["vendor"]
                        break

        pass_test = True

        # Heuristic mutates to {"$ne": excluded_vendor}
        if excluded_vendor and type(extracted_vendor_val) is dict:
            if extracted_vendor_val.get("$ne") != excluded_vendor:
                pass_test = False
        elif excluded_vendor and extracted_vendor_val == excluded_vendor:
            pass_test = False

        if expected_vendor and extracted_vendor_val != expected_vendor and type(extracted_vendor_val) is not dict:
            pass_test = False

        accuracy_score = 1 if pass_test else 0

        wandb.log({
            "query": query,
            "extraction_latency_ms": extraction_latency_ms,
            "filter_correct": accuracy_score
        })

        print(f"Latency: {extraction_latency_ms:5.1f}ms | Correct: {accuracy_score} | Filters: {chroma_filters} | Query: {query[:50]}...")

# Execute A/B Evaluation Loop
await evaluate_negation_routing()



--- Running Negation Benchmark (Heuristic Window) ---
Latency: 185.1ms | Correct: 1 | Filters: {'vendor': {'$ne': 'ZTE'}} | Query: BGP routes are constantly flapping on the core net...
Latency: 185.2ms | Correct: 1 | Filters: {'vendor': {'$ne': 'Cisco'}} | Query: The Cisco router is operating normally, but the DW...
Latency: 186.6ms | Correct: 1 | Filters: {'vendor': {'$ne': 'Juniper'}} | Query: How do I troubleshoot OSPF neighbor state stuck in...
Latency: 179.2ms | Correct: 1 | Filters: {'vendor': 'Cisco'} | Query: I am not looking for Nokia instructions right now....
Latency: 157.5ms | Correct: 1 | Filters: {'vendor': 'Juniper'} | Query: Fix corrupted BGP table on Juniper router....


In [14]:
import os
import asyncio
import json
import re
import pandas as pd
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from ragas.messages import HumanMessage, AIMessage


# 1. INIT GEMMA
def init_llm():
    return ChatGoogleGenerativeAI(
        model="gemma-3-27b-it",
        temperature=0.0,
        max_retries=5 # Added built-in LangChain retries
    )

multi_turn_tests = [
    {
        "combo_id": "MTS-ADV-010-AMF-DISTRACTION",
        "reference": "Correctly remembers the Ericsson AMF context from Turn 1, ignores the Ciena DWDM distraction from Turn 2, and successfully identifies the SMF as the 'confused_with' warning.",
        "conversation": [
            HumanMessage(content="We are seeing massive 5G UE registration failures across the network. Subscribers can't even attach. Our core runs entirely on Ericsson equipment. Please provide the mitigation steps."),
            AIMessage(content="REASONING: 5G UE registration failures on Ericsson equipment point to an AMF (Access and Mobility Function) failure. I need to provide the Ericsson-specific AMF troubleshooting steps.\nFINAL_ANSWER: 1. Log into the Ericsson AMF node. \n2. Verify the AMF service status using `amf_status_check` [SOP-AMF101]. \n3. Check for active critical alarms on the mobility layer [SOP-AMF101]."),
            HumanMessage(content="Hold on, the transport team just reported a critical fiber link failure on the Ciena ring. They suspect an optical wavelength mismatch. What are the steps to fix that?"),
            AIMessage(content="REASONING: The user is now asking about an optical wavelength mismatch on Ciena equipment. I must provide the specific SOP for Ciena DWDM issues.\nFINAL_ANSWER: 1. Access the Ciena transport node. \n2. Verify the optical transceiver status with `port show optical-power` [SOP-DWM202]. \n3. Confirm DWDM channel assignments are matching [SOP-DWM202]."),
            HumanMessage(content="Ignore the transport ring for now and let's focus strictly on the 5G registration failures. Give me the troubleshooting steps again, and explicitly tell me which function I should absolutely NOT confuse it with.")
        ]
    },
    {
        "combo_id": "MTS-ADV-011-RAN-DISTRACTION",
        "reference": "Correctly remembers the Nokia X2 Interface context from Turn 1, ignores the Fortinet NAT distraction from Turn 2, and successfully identifies the S1 Interface as the 'confused_with' warning.",
        "conversation": [
            HumanMessage(content="Active mobile handovers are failing between adjacent Nokia base stations, causing dropped calls when users drive. What are the troubleshooting steps?"),
            AIMessage(content="REASONING: Inter-cell handover failures on Nokia base stations indicate an X2 Interface issue. I need to provide the Nokia-specific X2 troubleshooting steps.\nFINAL_ANSWER: 1. Access the Nokia eNodeB management interface. \n2. Verify the X2 interface status using `show interface x2 detail` [SOP-RAN303]. \n3. Check for SCTP association drops [SOP-RAN303]."),
            HumanMessage(content="Wait, an enterprise client is complaining that internal users cannot reach the internet because of a full NAT table on their Fortinet firewall. Give me the steps for that."),
            AIMessage(content="REASONING: The user is now asking about NAT pool exhaustion on Fortinet equipment. I must provide the SOP for Fortinet NAT issues.\nFINAL_ANSWER: 1. Access the Fortinet firewall CLI. \n2. Check the active NAT session count with `get system session stat` [SOP-NAT404]. \n3. Clear stale NAT sessions if necessary [SOP-NAT404]."),
            HumanMessage(content="Let's drop the firewall ticket. Back to the dropped mobile calls on the towers. Provide the mitigation steps again, and warn me exactly which interface I must avoid bouncing by mistake.")
        ]
    },
    {
        "combo_id": "MTS-ADV-012-SPLITBRAIN-DISTRACTION",
        "reference": "Correctly remembers the Palo Alto Split-Brain context from Turn 1, ignores the Samsung VoLTE distraction from Turn 2, and successfully identifies HA Sync Failure as the 'confused_with' warning.",
        "conversation": [
            HumanMessage(content="Our Palo Alto edge firewalls are acting up. Both devices have assumed the Active role, causing massive IP conflicts and ARP poisoning. How do we fix this?"),
            AIMessage(content="REASONING: Both firewalls assuming the Active role on Palo Alto equipment is a Split-Brain HA scenario. I need to provide the Palo Alto Split-Brain troubleshooting steps.\nFINAL_ANSWER: 1. Log into the Palo Alto firewall management plane. \n2. Check the HA status on both nodes using `show high-availability all` [SOP-HA505]. \n3. Suspend the secondary node temporarily to stop ARP conflicts [SOP-HA505]."),
            HumanMessage(content="Just got an urgent alert about VoLTE call setup failures on the Samsung IMS platform. Provide the steps to troubleshoot this."),
            AIMessage(content="REASONING: The user is now asking about VoLTE call setup failures on Samsung IMS. I must provide the SOP for SIP INVITE failures.\nFINAL_ANSWER: 1. Access the Samsung IMS core. \n2. Trace the SIP signaling with `trace sip-invite user <impu>` [SOP-IMS606]. \n3. Verify the S-CSCF routing logic [SOP-IMS606]."),
            HumanMessage(content="Disregard the VoLTE alert, the voice team is handling it. Back to the edge firewalls causing IP conflicts. What are the steps, and what HA state is this commonly confused with?")
        ]
    },
    {
        "combo_id": "MTS-ADV-013-OPTICAL-DISTRACTION",
        "reference": "Correctly remembers the Cisco DWDM context from Turn 1, ignores the Juniper BGP distraction from Turn 2, and successfully identifies CWDM as the 'confused_with' warning.",
        "conversation": [
            HumanMessage(content="We have a critical fiber link failure on the Cisco Metro Fiber Ring. The optics team suspects an optical wavelength mismatch. Please provide the mitigation steps."),
            AIMessage(content="REASONING: An optical wavelength mismatch causing a link failure on Cisco equipment points to a DWDM issue. I need to provide the Cisco DWDM troubleshooting steps.\nFINAL_ANSWER: 1. Access the Cisco transport node. \n2. Check the DWDM interface configurations with `show hw-module interface optics detail` [SOP-OPT707]. \n3. Verify the tuned wavelength matches the remote end [SOP-OPT707]."),
            HumanMessage(content="Hold that thought, we are seeing iBGP route propagation issues on the Juniper backbone. What do I check?"),
            AIMessage(content="REASONING: The user is now asking about iBGP route propagation issues on Juniper equipment. I must provide the SOP for BGP Route Reflector issues.\nFINAL_ANSWER: 1. Access the Juniper backbone router. \n2. Verify the BGP neighbor state with `show bgp summary` [SOP-BGP808]. \n3. Check the route reflector client configuration [SOP-BGP808]."),
            HumanMessage(content="Scrap the routing issue, let's fix the fiber link first. Provide the optical troubleshooting steps and exactly which term I shouldn't mix it up with.")
        ]
    },
    {
        "combo_id": "MTS-ADV-014-VPN-DISTRACTION",
        "reference": "Correctly remembers the Check Point IPsec context from Turn 1, ignores the Huawei HSS distraction from Turn 2, and successfully identifies 'clear crypto isakmp' as the 'confused_with' warning.",
        "conversation": [
            HumanMessage(content="I need to drop all active data-plane IPsec VPN tunnels momentarily on the Check Point gateway. Please provide the exact steps to do this safely."),
            AIMessage(content="REASONING: Dropping active data-plane IPsec VPN tunnels on Check Point requires the specific command for clearing IPsec SAs. I need to provide the Check Point troubleshooting steps.\nFINAL_ANSWER: 1. Access the Check Point gateway via SSH. \n2. Enter expert mode [SOP-VPN909]. \n3. Execute the command to clear the specific IPsec SAs using `vpn tu` or the equivalent specific SA drop command [SOP-VPN909]."),
            HumanMessage(content="Also, we have a major subscriber authentication failure in the Huawei LTE/5G interworking core. Give me the steps for that."),
            AIMessage(content="REASONING: The user is now asking about subscriber authentication failures on Huawei equipment. I must provide the SOP for HSS issues.\nFINAL_ANSWER: 1. Log into the Huawei HSS node. \n2. Check the diameter link status towards the MME/AMF with `display diameter link all` [SOP-HSS111]. \n3. Verify subscriber profile provisioning [SOP-HSS111]."),
            HumanMessage(content="Let's deal with the mobile core later. Back to the VPN tunnels. Give me the steps to drop the active data-plane again, and explicitly state what command I should strictly avoid confusing it with.")
        ]
    }
]

# 3. ROBUST JSON PARSER
def extract_json(text):
    """Safely extracts JSON even if wrapped in Markdown formatting."""
    if not text: return None

    # Strip markdown backticks if present
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]

    text = text.strip()

    # Fallback to regex if there's conversational text around the JSON
    try:
        # Match outermost brackets
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON. Raw text: {text[:100]}...")
        return None

# 4. LLM JUDGE WITH CHAIN-OF-THOUGHT
async def score_all(llm, conversation, response, reference):
    convo = "\n".join([f"{'User' if isinstance(m, HumanMessage) else 'AI'}: {m.content}" for m in conversation])

    # Moved 'step_by_step_analysis' to the TOP of the JSON.
    # This forces the LLM to write out its logic before outputting the scores,
    # dramatically improving the accuracy of the evaluation.
    prompt = f"""
You are an expert telecom network engineer evaluating an AI assistant.
Evaluate the AI's final response based strictly on the provided context and reference.

Conversation History:
{convo}

Final Generated Response:
{response}

Reference Standard:
{reference}

Return ONLY valid JSON matching this exact schema:
{{
  "step_by_step_analysis": "Write 2-3 sentences analyzing if the AI found the correct root cause according to the reference.",
  "accuracy": 0-2,
  "topic": 0-2,
  "hallucination": 0-1,
  "reasoning": 0-2,
  "final_score": 0-10
}}

Scoring Guidelines:
- accuracy: 2 = perfect root cause, 1 = partial, 0 = incorrect
- topic: 2 = highly relevant telecom answer, 0 = off-topic
- hallucination: 1 = invented false commands/facts, 0 = factually clean
- reasoning: 2 = excellent multi-step logic, 0 = leaps to conclusions
"""

    # We rely on Langchain's built-in retries (configured in init_llm) + a fallback
    try:
        res = await llm.ainvoke(prompt)
        parsed = extract_json(res.content)
    except Exception as e:
        print(f"LLM API Error: {e}")
        parsed = None

    if not parsed:
        return {
            "step_by_step_analysis": "Eval Failed - Parse or API Error",
            "accuracy": 0, "topic": 0, "hallucination": 1,
            "reasoning": 0, "final_score": 0
        }
    return parsed

# 5. ROBUST RAG HISTORY FORMATTER
def format_history_for_rag(conversation_list):
    """Safely pairs queries and responses even if the sequence isn't perfect."""
    history = []
    current_query = None

    # Process all messages except the very last one (which is the current query)
    for msg in conversation_list[:-1]:
        if isinstance(msg, HumanMessage):
            current_query = msg.content
        elif isinstance(msg, AIMessage) and current_query:
            history.append({
                "query": current_query,
                "response": msg.content
            })
            current_query = None # Reset after a pair is made

    return history

# 6. SEMAPHORE CONTROLLED PROCESSING
async def process_case(llm, test, semaphore):
    async with semaphore: # Limits concurrent executions
        conversation = test["conversation"]
        final_query = conversation[-1].content
        formatted_history = format_history_for_rag(conversation)

        print(f"Testing Scenario: {test['combo_id']}...")
        ans, srcs = await process_query(final_query, formatted_history)

        scores = await score_all(llm, conversation, ans, test["reference"])

        return {
            "Scenario": test["combo_id"],
            "Generated_Answer": ans[:150] + "..." if len(ans) > 150 else ans,
            **scores
        }

# 7. PARALLEL EVALUATION RUN
async def evaluate_rag_pipeline(max_concurrent=3):
    llm = init_llm()
    print(f"Starting eval of {len(multi_turn_tests)} scenarios (Max Concurrent: {max_concurrent})...")

    # Use a semaphore to prevent crashing vLLM or hitting Gemini limits
    semaphore = asyncio.Semaphore(max_concurrent)

    tasks = [process_case(llm, t, semaphore) for t in multi_turn_tests]
    results = await asyncio.gather(*tasks)

    # Reorder columns for readability
    df = pd.DataFrame(results)
    cols = ['Scenario', 'final_score', 'accuracy', 'topic', 'hallucination', 'reasoning', 'Generated_Answer', 'step_by_step_analysis']
    return df[cols]

# Execute
df_multi_turn_eval = await evaluate_rag_pipeline(max_concurrent=3)
display(df_multi_turn_eval)

wandb.log({
    'multi_turn_final_score': df_multi_turn_eval['final_score'].mean(),
    'multi_turn_accuracy': df_multi_turn_eval['accuracy'].mean(),
    'multi_turn_topic': df_multi_turn_eval['topic'].mean(),
    'multi_turn_hallucination': df_multi_turn_eval['hallucination'].mean(),
    'multi_turn_reasoning': df_multi_turn_eval['reasoning'].mean()
})
wandb.finish()

Starting eval of 5 scenarios (Max Concurrent: 3)...
Testing Scenario: MTS-ADV-010-AMF-DISTRACTION...
  [Diagnostics] Router applied filters: None
Testing Scenario: MTS-ADV-011-RAN-DISTRACTION...
  [Diagnostics] Router applied filters: None
Testing Scenario: MTS-ADV-012-SPLITBRAIN-DISTRACTION...
  [Diagnostics] Router applied filters: None
Testing Scenario: MTS-ADV-013-OPTICAL-DISTRACTION...
  [Diagnostics] Router applied filters: None
Testing Scenario: MTS-ADV-014-VPN-DISTRACTION...
  [Diagnostics] Router applied filters: None


,Scenario,final_score,accuracy,topic,hallucination,reasoning,Generated_Answer,step_by_step_analysis
0,MTS-ADV-010-AMF-DISTRACTION,2,0,2,1,0,"To troubleshoot 5G registration failures, foll...",The AI completely disregarded the initial cont...
1,MTS-ADV-011-RAN-DISTRACTION,10,2,2,0,2,To resolve the issue of dropped mobile calls o...,The AI successfully pivoted back to the origin...
2,MTS-ADV-012-SPLITBRAIN-DISTRACTION,10,2,2,0,2,"To resolve the issue, follow the steps outline...",The AI correctly identifies the issue as a Spl...
3,MTS-ADV-013-OPTICAL-DISTRACTION,7,1,2,0,2,"To troubleshoot the fiber link, follow the ste...",The AI successfully returned to the original C...
4,MTS-ADV-014-VPN-DISTRACTION,7,1,2,0,1,To drop the active data-plane IPsec VPN tunnel...,The AI correctly returns to the Check Point VP...


context_precision,▁
context_recall,▁
extraction_latency_ms,███▆▁
faithfulness,▁
filter_correct,▁▁▁▁▁
multi_turn_accuracy,▁
multi_turn_final_score,▁
multi_turn_hallucination,▁
multi_turn_reasoning,▁
multi_turn_topic,▁
context_precision,0.86778


In [15]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic


In [16]:
import nest_asyncio
import uvicorn
import threading
import traceback
import sqlite3
import shutil
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok, conf
from google.colab import userdata

# Applying nest_asyncio so Uvicorn plays nicely with Colab's event loop
nest_asyncio.apply()

conn = sqlite3.connect(SQLITE_PATH, check_same_thread=False)
cursor = conn.cursor()

# 1. Define the API Schema
class QueryRequest(BaseModel):
    query: str
    history: list = []

class SopResponse(BaseModel):
    sop_id: str
    vendor: str
    severity: str
    title: str

class QueryResponse(BaseModel):
    answer: str
    retrieved_sops: list[SopResponse]

# 2. Initialize FastAPI
app = FastAPI(title="NetRestore RAG API", version="1.0")

# 3. Define the POST Endpoint
@app.post("/ask", response_model=QueryResponse)
async def ask_netrestore(request: QueryRequest):
    try:
        print(f"\nReceived query from frontend: {request.query}")

        # Call pipeline
        ans, srcs = await process_query(request.query, request.history)

        print("Query processed successfully! Sending back to frontend...")

        # Format the sources cleanly for the JSON response
        formatted_srcs = []
        for s in srcs:
            formatted_srcs.append(SopResponse(
                sop_id=s.get("sop_id", "N/A"),
                vendor=s.get("vendor", "Unknown"),
                severity=s.get("severity", "UNKNOWN"),
                title=s.get("title", "Untitled")
            ))

        return QueryResponse(answer=ans, retrieved_sops=formatted_srcs)

    except Exception as e:
        print("\nERROR IN PROCESS_QUERY:")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


# 4. Expose server to the internet

ngrok_exec = shutil.which("ngrok")
if not ngrok_exec:
    raise RuntimeError("Ngrok binary not found!")

pyngrok_config = conf.PyngrokConfig(ngrok_path=ngrok_exec)
conf.set_default(pyngrok_config)

ngrok_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_token, pyngrok_config=pyngrok_config)

# Close any existing tunnels to prevent errors
ngrok.kill()

public_url = ngrok.connect(8501, pyngrok_config=pyngrok_config).public_url

print(f"NETRESTORE API IS LIVE AT: {public_url}/ask")

# 5. Start the Server in a Background Thread
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8501)

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print("Background thread started. API is ready to receive requests!")



NETRESTORE API IS LIVE AT: https://unverbosely-ascocarpous-vickey.ngrok-free.dev/ask
Background thread started. API is ready to receive requests!


In [17]:
# Adding streaming endpoint to existing FastAPI app
@app.post("/ask-stream")
async def ask_netrestore_stream(request: QueryRequest):
    async def generate_stream():
        try:
            print(f"\n[Streaming] Received query: {request.query}")

            # Generate streaming response
            async for chunk in process_query_stream(request.query, request.history):
                yield f"data: {chunk}\n\n"

        except Exception as e:
            print(f"\n[Streaming] Error: {e}")
            error_chunk = json.dumps({"type": "error", "error": str(e)})
            yield f"data: {error_chunk}\n\n"

    return StreamingResponse(
        generate_stream(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "Access-Control-Allow-Origin": "*"
        }
    )

